# V2_Inference

In [1]:

# ============================================================
# CHART-TO-TABLE INFERENCE APP
# Frozen Donut encoder + pretrained Donut decoder (v2)
# ============================================================

import os
import re
import glob
import tempfile

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gradio as gr
import pandas as pd
import torch
import torch.nn as nn

from transformers import DonutProcessor, VisionEncoderDecoderModel


# ============================================================
# CHANGE ONLY THIS PATH WHEN USING ANOTHER V2 CHECKPOINT
# It can be a .pt file path OR a directory containing one .pt.
# ============================================================

CKPT_PATH = "/kaggle/input/datasets/rajakdisham/v2-supervised-ie643/decoder_v2_epoch1.pt"

ARCH = "v2"
DONUT = "naver-clova-ix/donut-base"

ROW_SPLIT = re.compile(r"\s*\[ROW\]\s*")
CELL_SPLIT = re.compile(r"\s*\|\s*")


# ============================================================
# RESOLVE CHECKPOINT PATH
# ============================================================

def resolve_checkpoint(path):
    path = os.path.abspath(path)

    if os.path.isfile(path):
        return path

    if os.path.isdir(path):
        candidates = sorted(
            glob.glob(os.path.join(path, "**", "*.pt"), recursive=True)
            + glob.glob(os.path.join(path, "**", "*.pth"), recursive=True)
        )

        # Remove duplicate paths
        candidates = list(dict.fromkeys(candidates))

        if len(candidates) == 1:
            return candidates[0]

        if len(candidates) == 0:
            raise FileNotFoundError(
                f"No .pt or .pth checkpoint found in: {path}"
            )

        print("Multiple checkpoints found:")
        for candidate in candidates:
            print(" ", candidate)

        raise ValueError(
            "Set CKPT_PATH to the full path of the checkpoint you want."
        )

    raise FileNotFoundError(f"Checkpoint path does not exist: {path}")


CKPT_PATH = resolve_checkpoint(CKPT_PATH)

print("Checkpoint:", CKPT_PATH)
print(f"Checkpoint size: {os.path.getsize(CKPT_PATH) / (1024**2):.1f} MB")


# ============================================================
# MODEL
# ============================================================

class DonutPretrainedDecoderModel(nn.Module):
    """Frozen Donut encoder + pretrained Donut decoder."""

    def __init__(self, donut, tokenizer):
        super().__init__()

        self.encoder = donut.encoder

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.decoder = donut.decoder
        c = self.decoder.config

        assert c.vocab_size == len(tokenizer), (
            f"Decoder vocab size {c.vocab_size} does not match "
            f"tokenizer size {len(tokenizer)}"
        )

        c.pad_token_id = tokenizer.pad_token_id
        c.bos_token_id = tokenizer.bos_token_id
        c.eos_token_id = tokenizer.eos_token_id
        c.decoder_start_token_id = tokenizer.bos_token_id
        c.use_cache = True


# ============================================================
# LOAD CHECKPOINT
# ============================================================

def load_model(ckpt_path, device):
    print("Loading Donut processor and pretrained model...")

    processor = DonutProcessor.from_pretrained(DONUT)
    donut = VisionEncoderDecoderModel.from_pretrained(DONUT)

    model = DonutPretrainedDecoderModel(donut, processor.tokenizer)

    print("Loading checkpoint weights...")

    ck = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False
    )

    if not isinstance(ck, dict):
        raise TypeError(
            "Unsupported checkpoint format: expected a dictionary."
        )

    if "decoder_state_dict" in ck:
        model.decoder.load_state_dict(ck["decoder_state_dict"])

    elif "model_state_dict" in ck:
        model.load_state_dict(ck["model_state_dict"])

    else:
        raise KeyError(
            "Checkpoint must contain 'decoder_state_dict' or "
            f"'model_state_dict'. Found keys: {list(ck.keys())[:30]}"
        )

    print(
        "Checkpoint loaded successfully.",
        "| Architecture:", ARCH,
        "| Validation loss:", ck.get("val_loss", "not stored"),
        "| Epoch:", ck.get("epoch", "not stored")
    )

    del ck, donut

    model = model.to(device).eval()
    return model, processor


# ============================================================
# GREEDY DECODING
# ============================================================

@torch.no_grad()
def generate(model, pixel_values, max_new_tokens=512,
             no_repeat_ngram=0):

    cfg = model.decoder.config

    encoder_output = model.encoder(pixel_values)
    encoder_hidden = encoder_output.last_hidden_state

    cur = torch.full(
        (1, 1),
        cfg.decoder_start_token_id,
        dtype=torch.long,
        device=pixel_values.device
    )

    past = None
    history = []
    seen = {}
    generated_ids = []
    hit_eos = False

    n = no_repeat_ngram

    for _ in range(max_new_tokens):
        out = model.decoder(
            input_ids=cur,
            encoder_hidden_states=encoder_hidden,
            past_key_values=past,
            use_cache=True,
            return_dict=True
        )

        past = out.past_key_values
        logits = out.logits[:, -1, :].float()

        # Optional repetition guard
        if n > 1 and len(history) >= n - 1:
            prefix = tuple(history[-(n - 1):])
            banned = seen.get(prefix, set())

            if banned:
                logits[0, list(banned)] = float("-inf")

        next_id = int(logits.argmax(dim=-1).item())

        if n > 1 and len(history) >= n - 1:
            prefix = tuple(history[-(n - 1):])
            seen.setdefault(prefix, set()).add(next_id)

        history.append(next_id)

        if next_id == cfg.eos_token_id:
            hit_eos = True
            break

        generated_ids.append(next_id)

        cur = torch.tensor(
            [[next_id]],
            dtype=torch.long,
            device=pixel_values.device
        )

    return generated_ids, hit_eos


# ============================================================
# PARSE GENERATED TABLE
# ============================================================

def parse_table(text):
    text = text.strip()

    if not text:
        return []

    return [
        CELL_SPLIT.split(row.strip())
        for row in ROW_SPLIT.split(text)
    ]


def to_dataframe(rows):
    if not rows:
        return pd.DataFrame()

    width = max(len(row) for row in rows)

    padded = [
        row + [""] * (width - len(row))
        for row in rows
    ]

    header = []
    seen = {}

    for j, value in enumerate(padded[0]):
        value = value or f"col{j + 1}"

        seen[value] = seen.get(value, 0) + 1

        if seen[value] > 1:
            value = f"{value} ({seen[value]})"

        header.append(value)

    return pd.DataFrame(padded[1:], columns=header)


# ============================================================
# GRADIO APP
# ============================================================

def build_app(model, processor, device):
    tokenizer = processor.tokenizer

    def run(image, use_guard, max_tokens):
        if image is None:
            return (
                "Upload a chart image first.",
                pd.DataFrame(),
                "",
                None
            )

        pixel_values = processor(
            image.convert("RGB"),
            return_tensors="pt"
        ).pixel_values.to(device)

        if device.type == "cuda":
            with torch.autocast("cuda", dtype=torch.float16):
                ids, hit_eos = generate(
                    model,
                    pixel_values,
                    max_new_tokens=int(max_tokens),
                    no_repeat_ngram=8 if use_guard else 0
                )
        else:
            ids, hit_eos = generate(
                model,
                pixel_values,
                max_new_tokens=int(max_tokens),
                no_repeat_ngram=8 if use_guard else 0
            )

        raw_text = tokenizer.decode(
            ids,
            skip_special_tokens=True
        ).strip()

        rows = parse_table(raw_text)
        dataframe = to_dataframe(rows)

        widths = sorted({len(row) for row in rows})

        info = (
            f"{len(ids)} generated tokens | "
            f"{len(rows)} rows | "
            f"row widths: {widths or '-'} | "
            f"{'Stopped at EOS' if hit_eos else 'Reached token limit'}"
        )

        csv_path = None

        if not dataframe.empty:
            with tempfile.NamedTemporaryFile(
                suffix=".csv",
                delete=False
            ) as tmp:
                csv_path = tmp.name

            dataframe.to_csv(csv_path, index=False)

        return raw_text, dataframe, info, csv_path

    with gr.Blocks(title="Chart-to-Table") as demo:
        gr.Markdown(
            "## Chart → Table\n"
            "Upload a chart image and generate a table. "
            "Verify the extracted values against the original chart."
        )

        with gr.Row():
            with gr.Column():
                image_input = gr.Image(
                    type="pil",
                    label="Chart image"
                )

                guard_input = gr.Checkbox(
                    value=False,
                    label="Repetition guard (may affect accuracy)"
                )

                max_tokens_input = gr.Slider(
                    minimum=64,
                    maximum=512,
                    value=512,
                    step=32,
                    label="Maximum generated tokens"
                )

                button = gr.Button(
                    "Generate table",
                    variant="primary"
                )

            with gr.Column():
                raw_output = gr.Textbox(
                    label="Raw generated sequence",
                    lines=6
                )

                table_output = gr.Dataframe(
                    label="Parsed table",
                    interactive=False
                )

                info_output = gr.Textbox(
                    label="Generation information"
                )

                csv_output = gr.File(
                    label="Download table as CSV"
                )

        button.click(
            fn=run,
            inputs=[
                image_input,
                guard_input,
                max_tokens_input
            ],
            outputs=[
                raw_output,
                table_output,
                info_output,
                csv_output
            ]
        )

    return demo


# ============================================================
# LAUNCH
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

model, processor = load_model(CKPT_PATH, device)

demo = build_app(model, processor, device)

demo.launch(share=True, debug=False)


Checkpoint: /kaggle/input/datasets/rajakdisham/v2-supervised-ie643/decoder_v2_epoch1.pt
Checkpoint size: 487.1 MB
Device: cuda
Loading Donut processor and pretrained model...


preprocessor_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.74k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/518 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.01M [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  809MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

Loading checkpoint weights...
Checkpoint loaded successfully. | Architecture: v2 | Validation loss: 0.4339563935535063 | Epoch: not stored
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://29444066aa50d59bbd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# V1 Inference 

In [2]:

# ============================================================
# V1 INFERENCE APP — Donut Encoder + 6-Layer BART Decoder
# ============================================================

import os
import re
import tempfile

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gradio as gr
import pandas as pd
import torch
import torch.nn as nn

from transformers import (
    DonutProcessor,
    VisionEncoderDecoderModel,
    BartConfig,
    BartForCausalLM,
)

# ============================================================
# 1. CONFIGURATION — CHANGE CHECKPOINT PATH HERE
# ============================================================

DONUT = "naver-clova-ix/donut-base"

CKPT_PATH = "/kaggle/input/datasets/rajakdisham/v1-final-pt-file/epoch_3.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_NEW_TOKENS = 512

ROW_SPLIT = re.compile(r"\s*\[ROW\]\s*")
CELL_SPLIT = re.compile(r"\s*\|\s*")


# ============================================================
# 2. V1 MODEL ARCHITECTURE
# Frozen Donut encoder + six-layer BART decoder
# ============================================================

class DonutBartModel(nn.Module):
    def __init__(self, donut, tokenizer):
        super().__init__()

        # Frozen visual encoder
        self.encoder = donut.encoder

        for parameter in self.encoder.parameters():
            parameter.requires_grad = False

        # BART decoder configuration
        config = BartConfig(
            vocab_size=len(tokenizer),
            d_model=1024,
            decoder_layers=6,
            decoder_attention_heads=8,
            decoder_ffn_dim=4096,
            max_position_embeddings=1024,
            pad_token_id=tokenizer.pad_token_id,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            decoder_start_token_id=tokenizer.bos_token_id,
            is_decoder=True,
            add_cross_attention=True,
            use_cache=True,
        )

        self.decoder = BartForCausalLM(config)


# ============================================================
# 3. LOAD V1 CHECKPOINT
# ============================================================

def load_model(ckpt_path, device):
    if not os.path.isfile(ckpt_path):
        raise FileNotFoundError(
            f"Checkpoint not found: {ckpt_path}\n"
            "Check the Kaggle input path."
        )

    print("Loading Donut processor...")
    processor = DonutProcessor.from_pretrained(DONUT)

    print("Loading Donut encoder...")
    donut = VisionEncoderDecoderModel.from_pretrained(DONUT)

    print("Building v1 architecture...")
    model = DonutBartModel(donut, processor.tokenizer)

    print("Loading checkpoint...")
    checkpoint = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False,
    )

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "decoder_state_dict" in checkpoint:
        state_dict = None
        model.decoder.load_state_dict(
            checkpoint["decoder_state_dict"],
            strict=True,
        )
    else:
        raise KeyError(
            "Checkpoint must contain 'model_state_dict' "
            "or 'decoder_state_dict'."
        )

    if state_dict is not None:
        model.load_state_dict(state_dict, strict=True)

    model = model.to(device)
    model.eval()

    print(f"Model loaded successfully on {device}")

    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    return model, processor


# ============================================================
# 4. AUTOREGRESSIVE GENERATION
# ============================================================

@torch.no_grad()
def generate(
    model,
    pixel_values,
    max_new_tokens=512,
    no_repeat_ngram=0,
):
    decoder_config = model.decoder.config

    # Encode image once
    encoder_outputs = model.encoder(pixel_values)
    encoder_hidden_states = encoder_outputs.last_hidden_state

    # Begin decoding
    current_token = torch.full(
        (1, 1),
        decoder_config.decoder_start_token_id,
        dtype=torch.long,
        device=pixel_values.device,
    )

    past_key_values = None
    generated_ids = []
    history = []
    seen_ngrams = {}
    hit_eos = False

    n = int(no_repeat_ngram)

    for _ in range(max_new_tokens):
        outputs = model.decoder(
            input_ids=current_token,
            encoder_hidden_states=encoder_hidden_states,
            past_key_values=past_key_values,
            use_cache=True,
            return_dict=True,
        )

        past_key_values = outputs.past_key_values
        logits = outputs.logits[:, -1, :].float()

        # Optional repeated n-gram blocking
        if n > 1 and len(history) >= n - 1:
            prefix = tuple(history[-(n - 1):])
            banned_tokens = seen_ngrams.get(prefix, set())

            if banned_tokens:
                logits[0, list(banned_tokens)] = float("-inf")

        next_token = int(torch.argmax(logits, dim=-1).item())

        if n > 1 and len(history) >= n - 1:
            prefix = tuple(history[-(n - 1):])
            seen_ngrams.setdefault(prefix, set()).add(next_token)

        history.append(next_token)

        if next_token == decoder_config.eos_token_id:
            hit_eos = True
            break

        generated_ids.append(next_token)

        current_token = torch.tensor(
            [[next_token]],
            dtype=torch.long,
            device=pixel_values.device,
        )

    return generated_ids, hit_eos


# ============================================================
# 5. PARSE GENERATED TABLE
# Expected format:
# header1 | header2 [ROW] value1 | value2
# ============================================================

def parse_table(text):
    text = text.strip()

    if not text:
        return []

    rows = ROW_SPLIT.split(text)

    return [
        CELL_SPLIT.split(row.strip())
        for row in rows
        if row.strip()
    ]


def to_dataframe(rows):
    if not rows:
        return pd.DataFrame()

    width = max(len(row) for row in rows)

    padded_rows = [
        row + [""] * (width - len(row))
        for row in rows
    ]

    header = []
    seen = {}

    for index, name in enumerate(padded_rows[0]):
        name = name.strip() or f"col{index + 1}"

        seen[name] = seen.get(name, 0) + 1

        if seen[name] == 1:
            header.append(name)
        else:
            header.append(f"{name} ({seen[name]})")

    return pd.DataFrame(padded_rows[1:], columns=header)


# ============================================================
# 6. INFERENCE FUNCTION
# ============================================================

def run_inference(image, max_tokens, block_repetition):
    if image is None:
        return (
            "",
            pd.DataFrame(),
            "Please upload a chart image.",
            None,
        )

    try:
        image = image.convert("RGB")

        pixel_values = processor(
            image,
            return_tensors="pt",
        ).pixel_values.to(DEVICE)

        use_amp = DEVICE.type == "cuda"

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            generated_ids, hit_eos = generate(
                model=model,
                pixel_values=pixel_values,
                max_new_tokens=int(max_tokens),
                no_repeat_ngram=8 if block_repetition else 0,
            )

        raw_text = processor.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        ).strip()

        rows = parse_table(raw_text)
        dataframe = to_dataframe(rows)

        status = (
            f"Architecture: V1 (Donut encoder + 6-layer BART decoder)\n"
            f"Device: {DEVICE}\n"
            f"Generated tokens: {len(generated_ids)}\n"
            f"EOS reached: {hit_eos}\n"
            f"Parsed rows including header: {len(rows)}\n"
            f"Parsed columns: "
            f"{len(rows[0]) if rows else 0}"
        )

        csv_path = None

        if not dataframe.empty:
            with tempfile.NamedTemporaryFile(
                mode="w",
                suffix=".csv",
                delete=False,
                encoding="utf-8-sig",
                newline="",
            ) as temp_file:
                csv_path = temp_file.name
                dataframe.to_csv(
                    temp_file,
                    index=False,
                )

        return raw_text, dataframe, status, csv_path

    except Exception as error:
        return (
            "",
            pd.DataFrame(),
            f"Inference error: {type(error).__name__}: {error}",
            None,
        )


# ============================================================
# 7. GRADIO INTERFACE
# ============================================================

def build_app():
    with gr.Blocks(title="Chart-to-Table — V1") as demo:
        gr.Markdown(
            "# Chart-to-Table — V1\n"
            "Upload a chart image to generate a structured table."
        )

        with gr.Row():
            with gr.Column():
                image_input = gr.Image(
                    type="pil",
                    label="Upload chart image",
                )

                max_tokens = gr.Slider(
                    minimum=64,
                    maximum=1024,
                    value=MAX_NEW_TOKENS,
                    step=64,
                    label="Maximum new tokens",
                )

                block_repetition = gr.Checkbox(
                    value=False,
                    label="Block repeated n-grams",
                )

                submit_button = gr.Button(
                    "Generate Table",
                    variant="primary",
                )

            with gr.Column():
                raw_output = gr.Textbox(
                    label="Raw generated output",
                    lines=8,
                )

                table_output = gr.Dataframe(
                    label="Parsed table",
                    interactive=False,
                )

                info_output = gr.Textbox(
                    label="Inference information",
                    lines=6,
                )

                csv_output = gr.File(
                    label="Download table as CSV",
                )

        submit_button.click(
            fn=run_inference,
            inputs=[
                image_input,
                max_tokens,
                block_repetition,
            ],
            outputs=[
                raw_output,
                table_output,
                info_output,
                csv_output,
            ],
        )

    return demo


# ============================================================
# 8. START APP
# ============================================================

model, processor = load_model(CKPT_PATH, DEVICE)

demo = build_app()

demo.launch(
    share=True,
    debug=False,
)


Loading Donut processor...
Loading Donut encoder...


Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

Building v1 architecture...
Loading checkpoint...
Model loaded successfully on cuda
GPU: Tesla T4
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://ad25b6eb5e92da1dfa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
